In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

X = pd.read_csv('X_feature.csv')
y = pd.read_csv('y_churn.csv')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=84)

scaler = StandardScaler()
scaler.set_output(transform='pandas')
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model_lr = LogisticRegression(class_weight='balanced')
model_lr.fit(X_train_scaled, y_train)
lr_predict = model_lr.predict(X_test_scaled)
lr_score = accuracy_score(y_test, lr_predict)
print(f"Logistics Scores: {lr_score:.3f}")
print("\nDetailed Report:\n", classification_report(y_test, lr_predict))
print(f"-> Model Coefficient {model_lr.coef_}")
cm = confusion_matrix(y_test, lr_predict)
print("\nThis is the Confusion Matrix\n", cm)

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)


xgb_model.fit(X_train_scaled, y_train)
xgb_predict = xgb_model.predict(X_test_scaled)
xgb_accuracy = accuracy_score(y_test, xgb_predict)
print(f"XGBoost Scores: {xgb_accuracy:.3f}")
print("\nDetailed Report:\n", classification_report(y_test, xgb_predict))
cm = confusion_matrix(y_test, xgb_predict)
print("\nThis is the Confusion Matrix\n", cm)

In [ ]:
plt.figure(figsize=(10, 6))
xgb.plot_importance(xgb_model, max_num_features=10, importance_type='weight')
plt.title("Most important features in XGBoost model")
plt.show()

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgb_model.fit(X_train_scaled, y_train)
lgb_predict = lgb_model.predict(X_test_scaled)
lgb_score = accuracy_score(y_test, lgb_predict)

print(f"\nLightGBM Score: {lgb_score:.3f}")
print(f"\nDetailed Report:\n{classification_report(y_test, lgb_predict)}")
cm = confusion_matrix(y_test, lgb_predict)
print("This is the confusion matrix\n", cm)

In [ ]:
plt.figure(figsize=(10, 6))
lgb.plot_importance(lgb_model, max_num_features=10, importance_type='split')
plt.title("Most important features in LightGBM model")
plt.show()

In [ ]:
xgb_param_dist = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': stats.uniform(0.01, 0.2),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'min_child_weight': [1, 3, 5, 7]
}

xgb_base = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric='logloss', random_state=42)

xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=15,
    scoring='f1',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train_scaled, y_train)
best_xgb = xgb_random.best_estimator_
xgb_tuned_predict = best_xgb.predict(X_test_scaled)

xgb_tuned_accuracy = accuracy_score(y_test, xgb_tuned_predict)
print(f"Accuracy Score: {xgb_tuned_accuracy:.3f}")
print("\n--- Best XGBoost Parameter Found ---")
print(xgb_random.best_params_)
print("\n--- Tuned XGBoosp Performance ---")
print(classification_report(y_test, xgb_tuned_predict))
print(confusion_matrix(y_test, xgb_tuned_predict))

In [ ]:
lgb_param_dist = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [3, 4, 5, 6, 7],
    'num_leaves': [15, 31, 63, 127],
    'learning_rate': stats.uniform(0.01, 0.2),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'min_child_weight': [10, 20, 30, 50]
}

lgb_base = lgb.LGBMClassifier(verbosity=-1, random_state=42)

lgb_random = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=15,
    scoring='f1',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

lgb_random.fit(X_train_scaled, y_train)
best_lgb = lgb_random.best_estimator_
lgb_tuned_predict = best_lgb.predict(X_test_scaled)

lgb_tuned_accuracy = accuracy_score(y_test, lgb_tuned_predict)
print(f"Accuracy Score: {lgb_tuned_accuracy:.3f}")
print("\n--- Best LightGB Parameter Found ---")
print(lgb_random.best_params_)
print("\n--- Tuned LightGB Performance ---")
print(classification_report(y_test, lgb_tuned_predict))
print(confusion_matrix(y_test, lgb_tuned_predict))

In [ ]:
import os
import joblib

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
# Logistics Regression
joblib.dump(model_lr, os.path.join(
    model_dir, "baseline_logistic_regression.pkl"))

# Tuned XGBoost model
joblib.dump(xgb_random.best_estimator_, os.path.join(
    model_dir, "tuned_xgboost.pkl"))
# Tuned LightGBM model
joblib.dump(lgb_random.best_estimator_, os.path.join(
    model_dir, "tuned_lightgbm.pkl"))
# Scaler
joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))

In [ ]:
X_train_scaled.head()